#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from tqdm import tqdm

/n/home07/than157/.conda/envs/llamafactory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load training set of hellaswag
dataset = load_dataset(path="qiaojin/PubMedQA", name="pqa_labeled", split="train")

     ###note
     # the train set is pqa_artficial (train split aka full dataset) -- ~211k samples
     # the test set is pqa_labeled (train set aka full dataset)-- ~1k samples

#print dataset info
print("Dataset info:")
print(dataset)

#convert to dataframe
df = dataset.to_pandas()

print("# samples:", df.shape[0])

df.head()


Dataset info:
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 1000
})
# samples: 1000


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


In [3]:
# write full question
# from learn-better/evolm/finetune/llama-factory/w_pubmedqa__create_sft_data.ipynb

def write_sft_input(row):
    #extract variables
    abstract_sentences = row['context']['contexts'].tolist()
    abstract_paragraph = " ".join(abstract_sentences)
    question = row['question']
    
    #write sft input
    sft_input = f'''Based on the paper abstract below, answer the following question with 'yes', 'maybe', or 'no'.

Question: {question}

Abstract: {abstract_paragraph}'''

    return sft_input




def write_sft_output(row):
    #extract variables
    long_answer = row['long_answer']
    final_decision = row['final_decision']

    #write sft output
    sft_output = f"The conclusion of the abstract is: {long_answer} Therefore, the answer is {final_decision}."

    return sft_output

In [4]:
#format input and output for all rows
df['sft_input'] = df.apply(write_sft_input, axis=1)
df['sft_output'] = df.apply(write_sft_output, axis=1)

In [5]:
#look at example
idx = 1
print('sft_input:')
print(df.iloc[idx]['sft_input'])

print('\nsft_output:')
print(df.iloc[idx]['sft_output'])

sft_input:
Based on the paper abstract below, answer the following question with 'yes', 'maybe', or 'no'.

Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?

Abstract: Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if their critical details appear under the same visual angle. Since optotypes are evaluated on individuals with good visual acuity and without eye disorders, differences in the lower visual acuity range cannot be excluded. In this study, visual acuity measured with the Snellen E was compared to the Landolt C acuity. 100 patients (age 8 - 90 years, median 60.5 years) with various eye disorders, among them 39 with amblyopia due to strabismus, and 13 healthy volunteers were tested. Charts with the Snellen E and the Landolt C (Precision Vision) which mimic the ETDRS charts were used to assess visual acuity. Three out of 5 optotypes per line had to be correctly ident

In [6]:
#check input lengths
df['sft_input_length'] = df["sft_input"].str.len()
df['sft_output_length'] = df["sft_output"].str.len()

#summary stats
print(df['sft_input_length'].describe())
print(df['sft_output_length'].describe())

count    1000.00000
mean     1553.46000
std       349.90832
min       490.00000
25%      1341.75000
50%      1555.00000
75%      1756.00000
max      2922.00000
Name: sft_input_length, dtype: float64
count    1000.000000
mean      333.248000
std       117.786621
min       127.000000
25%       246.000000
50%       316.000000
75%       401.250000
max       891.000000
Name: sft_output_length, dtype: float64


## save relevant columns in df to proper format

In [7]:
### create jsonl file

#format data for sft
data = []

# Write JSONL file
with open("data/cooked/pubmedqa.jsonl", "w", encoding="utf-8") as f:
    for idx in tqdm(range(df.shape[0])):
        row = df.iloc[idx]
        item = {
            "id": str(row["pubid"]),
            "problem": str(row["sft_input"]),
            "gt_solution": str(row["sft_output"]), #not used because gt_answer is not null
            "gt_answer": str(row["final_decision"])
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Final # of samples in json file:", len(data))
print("Complete!")

100%|██████████| 1000/1000 [00:00<00:00, 18973.17it/s]

Final # of samples in json file: 0
Complete!
